# 06 — Synthèse des Résultats

## Objectif

Rassembler tous les résultats des expériences précédentes et produire
des tableaux et graphiques prêts pour le mémoire.

**Ce notebook produit :**
1. Tableau récapitulatif de toutes les expériences
2. Graphiques comparatifs
3. Fichier CSV complet

---
**Pourquoi cette synthèse est importante :**
Le mémoire doit présenter des résultats clairs et reproductibles.
Ce notebook centralise l'ensemble des données expérimentales.

## 1. Imports

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for _ in range(4):
    if (ROOT / "src").exists():
        break
    ROOT = ROOT.parent

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

print("✅ Imports réussis")

## 2. Chargement de tous les résultats

On scanne le dossier `evaluation/results/` et on charge tous les CSV.

In [ ]:
results_dir = ROOT / "evaluation" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

# Charger tous les fichiers CSV
all_data = {}
for csv_file in sorted(results_dir.glob("*.csv")):
    name = csv_file.stem
    df = pd.read_csv(csv_file)
    all_data[name] = df
    print(f"📄 {name:40s} → {len(df)} lignes")

print(f"\n📚 {len(all_data)} fichier(s) chargé(s)")

## 3. Tableau récapitulatif

Pour chaque expérience, on extrait la moyenne des métriques.

In [ ]:
rows = []
for name, df in all_data.items():
    if "summary" in name:
        # Fichier summary déjà agrégé
        for _, row in df.iterrows():
            r = {"Expérience": name.replace("_summary", "")}
            for col in df.columns:
                if col not in ["Expérience", "Question", "Embedding", "Modèle", "Configuration", "Top-K"]:
                    try:
                        r[col] = round(float(row[col]), 4)
                    except (ValueError, TypeError):
                        pass
            rows.append(r)

if rows:
    summary = pd.DataFrame(rows)
    summary.set_index("Expérience", inplace=True)
    summary = summary.groupby("Expérience").mean().round(4)
    summary
else:
    print("Aucun résultat trouvé. Lancez d'abord les notebooks 02-05.")

## 4. Export du tableau récapitulatif

In [ ]:
if rows:
    csv_path = results_dir / "06_synthese_finale.csv"
    summary.to_csv(csv_path)
    print(f"✅ Exporté : {csv_path}")
    
    xlsx_path = results_dir / "06_synthese_finale.xlsx"
    summary.to_excel(xlsx_path)
    print(f"✅ Exporté : {xlsx_path}")

## 5. Graphique : Comparaison LLM

Barplot des scores par modèle LLM.

In [ ]:
llm_file = results_dir / "02_compare_llms.csv"
if llm_file.exists():
    df_llm = pd.read_csv(llm_file)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = df_llm.melt(id_vars=["Modèle"], value_vars=["Faithfulness", "AnswerRelevancy"],
                          var_name="Métrique", value_name="Score")
    sns.barplot(data=plot_df, x="Modèle", y="Score", hue="Métrique", ax=ax)
    ax.set_title("Comparaison des LLM", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    
    fig_path = ROOT / "evaluation" / "figures" / "compare_llms.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Figure sauvegardée : {fig_path}")
else:
    print("ℹ️  Lancez d'abord 02_compare_llms.ipynb")

## 6. Graphique : Impact de Top-K

Courbe d'évolution des métriques en fonction du nombre de chunks.

In [ ]:
topk_file = results_dir / "04_compare_topk_summary.csv"
if topk_file.exists():
    df_topk = pd.read_csv(topk_file)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_topk["Top-K"], df_topk["Faithfulness"], marker="o", linewidth=2, label="Fidélité")
    ax.plot(df_topk["Top-K"], df_topk["AnswerRelevancy"], marker="s", linewidth=2, label="Pertinence")
    ax.set_xlabel("Top-K")
    ax.set_ylabel("Score")
    ax.set_title("Impact de Top-K", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    
    fig_path = ROOT / "evaluation" / "figures" / "impact_topk.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Figure sauvegardée : {fig_path}")
else:
    print("ℹ️  Lancez d'abord 04_compare_retrieval.ipynb")

## 7. Graphique : Comparaison des embeddings

In [ ]:
emb_file = results_dir / "03_compare_embeddings.csv"
if emb_file.exists():
    df_emb = pd.read_csv(emb_file)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df = df_emb.melt(id_vars=["Embedding"], value_vars=["Faithfulness", "AnswerRelevancy"],
                          var_name="Métrique", value_name="Score")
    sns.barplot(data=plot_df, x="Embedding", y="Score", hue="Métrique", ax=ax)
    ax.set_title("Comparaison des Embeddings", fontsize=14, fontweight="bold")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    
    fig_path = ROOT / "evaluation" / "figures" / "compare_embeddings.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Figure sauvegardée : {fig_path}")
else:
    print("ℹ️  Lancez d'abord 03_compare_embeddings.ipynb")

## 8. Fichiers produits

Tout le contenu généré est dans les dossiers :
- `evaluation/results/` → données CSV/Excel
- `evaluation/figures/` → graphiques PNG

In [ ]:
print("📂 Résultats disponibles :")
for p in sorted(results_dir.glob("*")):
    print(f"   📄 {p.name}")

fig_dir = ROOT / "evaluation" / "figures"
if fig_dir.exists():
    print(f"\n🖼️  Figures disponibles :")
    for p in sorted(fig_dir.glob("*")):
        print(f"   🖼️  {p.name}")

## Conclusion

Ce notebook a produit l'ensemble des éléments nécessaires au mémoire :

| Élément | Emplacement |
|---|---|
| Tableau récapitulatif | `evaluation/results/06_synthese_finale.csv` |
| Graphique LLM | `evaluation/figures/compare_llms.png` |
| Graphique embeddings | `evaluation/figures/compare_embeddings.png` |
| Graphique Top-K | `evaluation/figures/impact_topk.png` |

**Prochaine étape :** intégrer ces résultats dans votre mémoire.